# 10 — Opti, SQP, and quadratic-program interfaces

**Objectives.** Express the same NLP readably with `Opti`, change parameters without rebuilding, and understand where QP plugins fit. `Opti` is a modeling layer over CasADi functions and NLP solvers; it does not change the mathematics.


In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Express an optimization problem with the Opti modeling layer

`ca.Opti` provides a higher-level interface for declaring variables,
parameters, objectives, and constraints. `set_value` supplies parameter
data without rebuilding the graph, while `set_initial` supplies only a
solver starting point. `ca.sumsqr` constructs a scalar sum of squared
elements and is used here for the circular constraint.

The model is

$$
\begin{aligned}
\min_x\quad &(x_0-t)^2+(x_1-1)^2\\
\text{s.t.}\quad &x_0\ge0,\;x_1\ge0,\\
&x_0+x_1\ge1,\\
&x_0^2+x_1^2\le4,
\end{aligned}
$$

where $t$ is an Opti parameter set to $2$ before solving. Although the
objective is quadratic, the quadratic inequality makes this a convenient
demonstration of the general nonlinear Opti/IPOPT workflow.


In [2]:
import casadi as ca

optimization = ca.Opti()
x = optimization.variable(2)
target = optimization.parameter()

optimization.minimize((x[0] - target) ** 2 + (x[1] - 1) ** 2)
optimization.subject_to(x >= 0)
optimization.subject_to(x[0] + x[1] >= 1)
optimization.subject_to(ca.sumsqr(x) <= 4)

optimization.set_value(target, 2.0)
optimization.set_initial(x, [0.5, 0.5])
optimization.solver(
    "ipopt",
    {"print_time": False},
    {"print_level": 0},
)

solution = optimization.solve()
print(
    "optimal x:",
    solution.value(x),
    "status:",
    solution.stats()["return_status"],
)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

optimal x: [1.78885439 0.89442719] status: Solve_Succeeded


### Inspect available conic solver interfaces

CasADi solver plugins are compiled independently. `CasadiMeta.plugins()`
reports the available backends using `Family::name` strings, so filtering
the `Conic` family reveals which quadratic/conic solvers can be selected in
the current environment. `ca.conic_in()` lists the standardized fields
accepted by CasADi's low-level conic interface.

A common quadratic-program form is

$$
\begin{aligned}
\min_x\quad &\tfrac12x^\mathsf{T}Hx+g^\mathsf{T}x\\
\text{s.t.}\quad &\ell_a\le Ax\le u_a,\\
&\ell_x\le x\le u_x.
\end{aligned}
$$

This cell does not solve that problem; it inspects which plugin backends
and standardized inputs would be available to do so.


In [3]:
# CasadiMeta.plugins reports compiled solver backends as Family::name.
installed_plugins = ca.CasadiMeta.plugins().split(";")
conic_plugins = [plugin for plugin in installed_plugins if plugin.startswith("Conic::")]
print("installed conic/QP plugins:", conic_plugins)

# conic_in lists the standardized input fields accepted by conic solvers.
print("conic input fields:", ca.conic_in())

installed conic/QP plugins: ['Conic::qpoases', 'Conic::fatrop', 'Conic::proxqp', 'Conic::osqp', 'Conic::nlpsol', 'Conic::qrqp', 'Conic::ipqp']
conic input fields: ['h', 'g', 'a', 'lba', 'uba', 'lbx', 'ubx', 'x0', 'lam_x0', 'lam_a0', 'q', 'p']


A QP has a quadratic objective and linear constraints. CasADi exposes QP solvers through `conic`/`qpsol`; availability varies by binary and license. `sqpmethod` repeatedly builds QP approximations to an NLP and is useful for comparison, while IPOPT is this course's primary backend.

`opti.debug.value(expr, opti.initial())`, `opti.debug.show_infeasibilities()`, `x_describe`, and `g_describe` connect algebra back to source. **Recap:** use `Opti` for readable modeling and raw `nlpsol` when exact vector ordering or generated interfaces matter. [Exercises](../exercises/04_nlp_debugging.ipynb).
